In [1]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

Current working directory:
C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\notebooks


In [2]:
from pathlib import Path

# Find the raw-data folder from the current directory
candidates = [
    Path.cwd() / "data" / "raw",
    Path.cwd().parent / "data" / "raw",
]

raw_dir = next(
    (path for path in candidates if path.is_dir()),
    None
)

if raw_dir is None:
    raise FileNotFoundError(
        "Could not find data/raw. Check your project folder."
    )

excel_files = sorted(
    list(raw_dir.glob("*.xlsx"))
    + list(raw_dir.glob("*.xls"))
)

print(f"Raw data folder: {raw_dir}")
print(f"Excel files found: {len(excel_files)}")

for file in excel_files:
    print(f"{file.name} — {file.stat().st_size / 1_000_000:.2f} MB")

Raw data folder: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\data\raw
Excel files found: 3
IDS-data-2011_17-May24.xlsx — 73.57 MB
IDS-data-2018-2022.xlsx — 65.36 MB
IDS-data-2023-2026.xlsx — 47.40 MB


In [3]:
import pandas as pd

In [4]:
file_path = excel_files[0]

# Inspect the workbook's sheet names
xls = pd.ExcelFile(file_path)

print("File:", file_path.name)
print("Sheets:", xls.sheet_names)

File: IDS-data-2011_17-May24.xlsx
Sheets: ['2011', '2012', '2013', '2014', '2015', '2016', '2017', 'Drug list and Measurement Units']


In [5]:
df = pd.read_excel(
    file_path,
    sheet_name=xls.sheet_names[0],
    nrows=5
)

print("Columns:")
print(df.columns.tolist())

display(df)

Columns:
['Seizure Date', 'Country/Territory of Seizure', 'ISO3', 'Subregion', 'Region', 'Drug/Substance', 'Quantity Seized', 'Measurement Unit', 'City', 'Administrative Region', 'Physical Seizure Location', 'Trafficking Mode of Transportation', 'Source', 'Reporting Channel']


,Seizure Date,Country/Territory of Seizure,ISO3,Subregion,Region,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Physical Seizure Location,Trafficking Mode of Transportation,Source,Reporting Channel
0,2011-12-31,Belgium,BEL,Western and Central Europe,Europe,Non-specified cocaine-type,2.00,kg,Brussels,NaN,Airport,Airplane,Government,Individual Drug Seizures (IDS)
1,2011-12-31,Bolivia (Plurinational State of),BOL,South America,Americas,Cocaine base,1.20,g,La Paz,NaN,Data not available,Data not available,Government,Individual Drug Seizures (IDS)
2,2011-12-31,Bolivia (Plurinational State of),BOL,South America,Americas,Cocaine base,2837.00,g,Cochabamba,NaN,Data not available,Data not available,Government,Individual Drug Seizures (IDS)
3,2011-12-31,Bolivia (Plurinational State of),BOL,South America,Americas,"Cocaine hydrochloride (HCl, powder cocaine)",4905.00,g,Cochabamba,NaN,Data not available,Data not available,Government,Individual Drug Seizures (IDS)
4,2011-12-31,Colombia,COL,South America,Americas,Cannabis herb (marijuana),2.33,kg,Bogota D.C.,NaN,Road/ Street/highway,Car/motorcycle,Government,Individual Drug Seizures (IDS)


1. Do all three workbooks use the same columns?
2. Do their sheets contain the same types of records?
3. Are dates and quantities stored consistently?
4. How much information is missing, particularly for transportation and location?

In [6]:
for file_path in excel_files:
    print(f"\n{'=' * 70}")
    print(f"FILE: {file_path.name}")

    xls = pd.ExcelFile(file_path)

    for sheet in xls.sheet_names:
        # Read column headers only, not the full data
        headers = pd.read_excel(
            xls,
            sheet_name=sheet,
            nrows=0
        ).columns.tolist()

        print(f"\nSheet: {sheet}")
        print(f"Number of columns: {len(headers)}")
        print(headers)

    xls.close()


FILE: IDS-data-2011_17-May24.xlsx

Sheet: 2011
Number of columns: 14
['Seizure Date', 'Country/Territory of Seizure', 'ISO3', 'Subregion', 'Region', 'Drug/Substance', 'Quantity Seized', 'Measurement Unit', 'City', 'Administrative Region', 'Physical Seizure Location', 'Trafficking Mode of Transportation', 'Source', 'Reporting Channel']

Sheet: 2012
Number of columns: 14
['Seizure Date', 'Country/Territory of Seizure', 'ISO3', 'Subregion', 'Region', 'Drug/Substance', 'Quantity Seized', 'Measurement Unit', 'City', 'Administrative Region', 'Physical Seizure Location', 'Trafficking Mode of Transportation', 'Source', 'Reporting Channel']

Sheet: 2013
Number of columns: 14
['Seizure Date', 'Country/Territory of Seizure', 'ISO3', 'Subregion', 'Region', 'Drug/Substance', 'Quantity Seized', 'Measurement Unit', 'City', 'Administrative Region', 'Physical Seizure Location', 'Trafficking Mode of Transportation', 'Source', 'Reporting Channel']

Sheet: 2014
Number of columns: 14
['Seizure Date', 'Cou

In [7]:
file_path = raw_dir / "IDS-data-2011_17-May24.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name="2011"
)

# Dataset size
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

# Data types and non-null counts
df.info()

Number of rows: 41806
Number of columns: 14
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41806 entries, 0 to 41805
Data columns (total 14 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Seizure Date                        41806 non-null  datetime64[ns]
 1   Country/Territory of Seizure        41806 non-null  object        
 2   ISO3                                41806 non-null  object        
 3   Subregion                           41806 non-null  object        
 4   Region                              41806 non-null  object        
 5   Drug/Substance                      41806 non-null  object        
 6   Quantity Seized                     41806 non-null  float64       
 7   Measurement Unit                    41806 non-null  object        
 8   City                                40288 non-null  object        
 9   Administrative Region               2340 non-null 

In [8]:
# Missing values by column
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
})

missing = missing.sort_values(
    "missing_pct",
    ascending=False
)

display(missing)

,missing_count,missing_pct,dtype
Administrative Region,39466,94.40,object
City,1518,3.63,object
Seizure Date,0,0.00,datetime64[ns]
Country/Territory of Seizure,0,0.00,object
ISO3,0,0.00,object
Subregion,0,0.00,object
Region,0,0.00,object
Drug/Substance,0,0.00,object
Quantity Seized,0,0.00,float64
Measurement Unit,0,0.00,object


In [9]:
# Check how dates are represented
print("Date examples:")
print(df["Seizure Date"].head(10).to_list())

# Check the quantity and measurement-unit fields
print("\nQuantity examples:")
print(df["Quantity Seized"].head(10).to_list())

print("\nMeasurement units:")
print(df["Measurement Unit"].value_counts(dropna=False).head(20))

# Check transport information
print("\nTransportation modes:")
print(
    df["Trafficking Mode of Transportation"]
    .value_counts(dropna=False)
    .head(20)
)

# Check reporting locations
location_cols = [
    "City",
    "Administrative Region",
    "Physical Seizure Location"
]

print("\nLocation completeness:")
display(df[location_cols].notna().mean().mul(100).round(2))

Date examples:
[Timestamp('2011-12-31 00:00:00'), Timestamp('2011-12-31 00:00:00'), Timestamp('2011-12-31 00:00:00'), Timestamp('2011-12-31 00:00:00'), Timestamp('2011-12-31 00:00:00'), Timestamp('2011-12-31 00:00:00'), Timestamp('2011-12-31 00:00:00'), Timestamp('2011-12-31 00:00:00'), Timestamp('2011-12-31 00:00:00'), Timestamp('2011-12-31 00:00:00')]

Quantity examples:
[2.0, 1.2, 2837.0, 4905.0, 2.33, 10.053, 250.0, 0.13, 0.5, 3.0]

Measurement units:
Measurement Unit
kg             29141
g              11864
unit             357
tablet/pill      333
plant             94
lt                14
Ampoule            2
other              1
Name: count, dtype: int64

Transportation modes:
Trafficking Mode of Transportation
Data not available            27543
Car/motorcycle                 6702
Other                          3636
Airplane                       2787
Vessel/boat                     905
Train/Bus                       231
Postal/Mail/Express Parcel        1
Bicycle/on foot/wal

City                          96.37
Administrative Region          5.60
Physical Seizure Location    100.00
dtype: float64

# Reliable data inventory